# Darwin API - Getting Started

This notebook demonstrates how to use the Darwin API for file transfer operations with AI agent integration.

## Features Covered
- Creating file transfer sessions
- Uploading and downloading files
- Listing remote directories
- MCP tool calling
- Agent coordination

## Setup

First, install required packages and import libraries:

In [ ]:
!pip install httpx jupyter-client ipywidgets

import httpx
import json
from typing import Dict, Any
from IPython.display import display, Markdown, JSON

## Configuration

Set up the API endpoint and credentials:

In [ ]:
# Darwin API Configuration
API_BASE_URL = "http://localhost:8000"
API_VERSION = "v1"

# Server credentials (update with your values)
SERVER_CONFIG = {
    "protocol": "sftp",
    "hostname": "example.com",
    "username": "user",
    "password": "password",
    "port": 22,
    "ssh_host_key_fingerprint": "ssh-rsa 2048 xx:xx:xx..."
}

# Create HTTP client
client = httpx.Client(base_url=API_BASE_URL, timeout=30.0)

## 1. Create a Session

Create a new file transfer session:

In [ ]:
# Create session
response = client.post(
    f"/api/{API_VERSION}/session/create",
    json={
        "options": SERVER_CONFIG,
        "session_name": "My Darwin Session"
    }
)

session_data = response.json()
session_id = session_data["session_id"]

print(f"✅ Session created: {session_id}")
display(JSON(session_data))

## 2. List Remote Directory

List files in a remote directory:

In [ ]:
# List directory
response = client.post(
    f"/api/{API_VERSION}/session/{session_id}/list",
    json={"remote_path": "/"}
)

files = response.json()
print(f"📁 Found {files['count']} files/directories")
display(JSON(files))

## 3. Upload Files

Upload a file to the remote server:

In [ ]:
# Upload file
response = client.post(
    f"/api/{API_VERSION}/session/{session_id}/upload",
    json={
        "local_path": "/path/to/local/file.txt",
        "remote_path": "/remote/destination/",
        "options": {
            "transfer_mode": "binary",
            "preserve_timestamp": True
        }
    }
)

result = response.json()
print(f"⬆️ Upload result: {result['message']}")
display(JSON(result))

## 4. Download Files

Download a file from the remote server:

In [ ]:
# Download file
response = client.post(
    f"/api/{API_VERSION}/session/{session_id}/download",
    json={
        "remote_path": "/remote/file.txt",
        "local_path": "/path/to/local/destination/",
        "options": {
            "transfer_mode": "binary"
        }
    }
)

result = response.json()
print(f"⬇️ Download result: {result['message']}")
display(JSON(result))

## 5. MCP Tool Calling

Use MCP (Model Context Protocol) for LLM integration:

In [ ]:
# List available MCP tools
response = client.get("/mcp/tools")
tools = response.json()

print(f"🛠️ Available MCP tools: {len(tools)}")
for tool in tools:
    print(f"  - {tool['name']}: {tool['description']}")

In [ ]:
# Call MCP tool
response = client.post(
    "/mcp/tools/call",
    json={
        "tool_name": "darwin_list_directory",
        "arguments": {
            "session_id": session_id,
            "remote_path": "/"
        }
    }
)

result = response.json()
print(f"✅ MCP tool call successful: {result['success']}")
display(JSON(result))

## 6. Agent Swarm Coordination

Create and coordinate multiple agents:

In [ ]:
# Create agents
agents = []
for i in range(3):
    response = client.post(
        f"/api/{API_VERSION}/agents",
        json={
            "name": f"FileTransferAgent-{i}",
            "agent_type": "file_transfer",
            "capabilities": ["upload", "download", "sync"],
            "config": {"max_concurrent_transfers": 5}
        }
    )
    agent_data = response.json()
    agents.append(agent_data["agent_id"])
    print(f"🤖 Created agent: {agent_data['name']} ({agent_data['agent_id']})")

print(f"\n✅ Created {len(agents)} agents")

In [ ]:
# Coordinate agent swarm
response = client.post(
    f"/api/{API_VERSION}/agents/swarm/coordinate",
    json={
        "task_description": "Upload files from multiple sources to remote server",
        "agents": agents,
        "config": {
            "parallel_execution": True,
            "timeout_seconds": 300
        }
    }
)

coordination = response.json()
print(f"🎯 Swarm coordination: {coordination['status']}")
print(f"   Coordination ID: {coordination['coordination_id']}")
print(f"   Agents assigned: {coordination['agents_assigned']}")
display(JSON(coordination))

## 7. Cleanup

Close the session and cleanup resources:

In [ ]:
# Close session
response = client.delete(f"/api/{API_VERSION}/session/{session_id}")
print(f"✅ Session {session_id} closed")

# Terminate agents
for agent_id in agents:
    response = client.delete(f"/api/{API_VERSION}/agents/{agent_id}")
    print(f"✅ Agent {agent_id} terminated")

# Close HTTP client
client.close()
print("\n🎉 Cleanup complete!")

## Next Steps

- Explore advanced transfer options (resume, synchronization)
- Build custom agent types for specific workflows
- Integrate with LLMs using MCP tools
- Create automated file processing pipelines

## Documentation

- API Documentation: http://localhost:8000/docs
- GitHub: https://github.com/darbotlabs/darwin
- MCP Specification: https://modelcontextprotocol.io